In [28]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

In [29]:
# Constants
NUM_CLASSES = 2
RANDOM_STATE = 1082
TRAIN_SIZE = 0.2
REAL_DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
BASE_DIR = "../../../Datasets/csv/UAVATTACK/"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

# Hyperparameters
INPUT_SIZE = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

models_list = ["ctgan", "gc", "tvae"]
sample_sizes = [10_000, 100_000, 1_000_000]

In [30]:
# Load real data (for testing)
real_df = pd.read_csv(REAL_DATA_PATH)
real_df.columns = real_df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
if 'label' not in real_df.columns:
    raise ValueError("Real data does not have 'label' column.")
X_real = real_df.drop(columns=[col for col in ['label', 'timestamp'] if col in real_df.columns])
y_real = real_df['label']

In [31]:
# Encode labels if needed
if y_real.dtype == object or y_real.dtype.name == 'category':
    le_real = LabelEncoder()
    y_real = le_real.fit_transform(y_real)
else:
    le_real = None

# Pre-scale real data (scaler will be fit on synthetic train)
X_real = X_real.astype('float32')

In [32]:
# Define LSTM Classifier
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [33]:
results = []

In [34]:
# Define a basic LSTM model for classification
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [35]:
for model_name in models_list:
    for n_samples in sample_sizes:
        file_name = f"spoofing_{model_name}_{n_samples}.csv"
        file_path = os.path.join(BASE_DIR, file_name)
        if not os.path.isfile(file_path):
            print(f"File {file_name} not found, skipping.")
            continue

        # Load synthetic dataset
        synth_df = pd.read_csv(file_path)
        if 'label' not in synth_df.columns:
            print(f"Skipping {file_name}: 'label' column not found.")
            continue

        X_synth = synth_df.drop('label', axis=1)
        y_synth = synth_df['label']

        # Encode synthetic labels if needed
        if y_synth.dtype == object or y_synth.dtype.name == 'category':
            le_synth = LabelEncoder()
            y_synth = le_synth.fit_transform(y_synth)

        # Check feature match
        if list(X_synth.columns) != list(X_real.columns):
            print(f"Feature mismatch between {file_name} and real data, skipping.")
            continue

        # Train/test split on synthetic
        X_train, _, y_train, _ = train_test_split(
            X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_real_scaled = scaler.transform(X_real)

        # Reshape for LSTM
        SEQUENCE_LENGTH = X_train_scaled.shape[1]
        X_train_lstm = X_train_scaled.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
        X_real_lstm = X_real_scaled.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)

        # Prepare tensors
        X_train_tensor = torch.tensor(X_train_lstm, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
        X_test_tensor = torch.tensor(X_real_lstm, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_real, dtype=torch.long)

        # DataLoaders
        train_data = TensorDataset(X_train_tensor, y_train_tensor)
        test_data = TensorDataset(X_test_tensor, y_test_tensor)
        train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

        # Init model, loss, optimizer
        model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # Training
        start_time = time.time()
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * X_batch.size(0)
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)
            if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == NUM_EPOCHS - 1:
                epoch_loss = running_loss / len(train_loader.dataset)
                epoch_acc = correct / total
                print(f"{file_name} | Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

        train_time = time.time() - start_time

        # Evaluation
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                outputs = model(X_batch)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        print(f"{file_name} | Accuracy: {acc:.4f} | F1: {f1:.4f}")

        # Store results
        results.append({
            "model": "LSTM Classifier",
            "dataset": file_name,
            "f1score": f1,
            "accuracy": acc,
            "train_time_sec": train_time,
            "parameters": str(model.state_dict().keys()),
            "num_epochs": NUM_EPOCHS
        })

spoofing_ctgan_10000.csv | Epoch 1/100, Loss: 0.6727, Acc: 0.5785
spoofing_ctgan_10000.csv | Epoch 10/100, Loss: 0.1362, Acc: 0.9585
spoofing_ctgan_10000.csv | Epoch 20/100, Loss: 0.1153, Acc: 0.9620
spoofing_ctgan_10000.csv | Epoch 30/100, Loss: 0.1187, Acc: 0.9645
spoofing_ctgan_10000.csv | Epoch 40/100, Loss: 0.0954, Acc: 0.9720
spoofing_ctgan_10000.csv | Epoch 50/100, Loss: 0.0731, Acc: 0.9785
spoofing_ctgan_10000.csv | Epoch 60/100, Loss: 0.0670, Acc: 0.9785
spoofing_ctgan_10000.csv | Epoch 70/100, Loss: 0.0646, Acc: 0.9800
spoofing_ctgan_10000.csv | Epoch 80/100, Loss: 0.0455, Acc: 0.9840
spoofing_ctgan_10000.csv | Epoch 90/100, Loss: 0.0474, Acc: 0.9840
spoofing_ctgan_10000.csv | Epoch 100/100, Loss: 0.0361, Acc: 0.9880
spoofing_ctgan_10000.csv | Accuracy: 0.9931 | F1: 0.9930
spoofing_ctgan_100000.csv | Epoch 1/100, Loss: 0.2919, Acc: 0.8578
spoofing_ctgan_100000.csv | Epoch 10/100, Loss: 0.0875, Acc: 0.9739
spoofing_ctgan_100000.csv | Epoch 20/100, Loss: 0.0643, Acc: 0.9811
spo

In [36]:
# Save results to CSV
results_df = pd.DataFrame(results)
file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../UAVATTACK_Results.csv
